In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import Modules

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

In [3]:
import warnings
warnings.filterwarnings('ignore')


# Load The Dataset

In [4]:
train = pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')

# Data Understanding

In [5]:
train.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert


In [6]:
test.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
0,18524,3.0,No,7.0,4.0,No,6.0,NaN
1,18525,NaN,Yes,0.0,0.0,Yes,5.0,1.0
2,18526,3.0,No,5.0,6.0,No,15.0,9.0
3,18527,3.0,No,4.0,4.0,No,5.0,6.0
4,18528,9.0,Yes,1.0,2.0,Yes,1.0,1.0


In [7]:
train.shape

(18524, 9)

In [8]:
test.shape

(6175, 8)

In [9]:
train['Personality'].unique()

array(['Extrovert', 'Introvert'], dtype=object)

In [10]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18524 entries, 0 to 18523
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         18524 non-null  int64  
 1   Time_spent_Alone           17334 non-null  float64
 2   Stage_fear                 16631 non-null  object 
 3   Social_event_attendance    17344 non-null  float64
 4   Going_outside              17058 non-null  float64
 5   Drained_after_socializing  17375 non-null  object 
 6   Friends_circle_size        17470 non-null  float64
 7   Post_frequency             17260 non-null  float64
 8   Personality                18524 non-null  object 
dtypes: float64(5), int64(1), object(3)
memory usage: 1.3+ MB


In [11]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6175 entries, 0 to 6174
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         6175 non-null   int64  
 1   Time_spent_Alone           5750 non-null   float64
 2   Stage_fear                 5577 non-null   object 
 3   Social_event_attendance    5778 non-null   float64
 4   Going_outside              5709 non-null   float64
 5   Drained_after_socializing  5743 non-null   object 
 6   Friends_circle_size        5825 non-null   float64
 7   Post_frequency             5767 non-null   float64
dtypes: float64(5), int64(1), object(2)
memory usage: 386.1+ KB


In [12]:
train.isnull().sum()

id                              0
Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
Personality                     0
dtype: int64

In [13]:
test.isnull().sum()

id                             0
Time_spent_Alone             425
Stage_fear                   598
Social_event_attendance      397
Going_outside                466
Drained_after_socializing    432
Friends_circle_size          350
Post_frequency               408
dtype: int64

In [14]:
train['Stage_fear'].unique()

array(['No', 'Yes', nan], dtype=object)

In [15]:
train.columns

Index(['id', 'Time_spent_Alone', 'Stage_fear', 'Social_event_attendance',
       'Going_outside', 'Drained_after_socializing', 'Friends_circle_size',
       'Post_frequency', 'Personality'],
      dtype='object')

In [16]:
train.duplicated().sum()

0

In [17]:
test.duplicated().sum()

0

# Data Preprocessing

In [18]:
num_cols = ['Time_spent_Alone','Social_event_attendance', 'Going_outside', 'Friends_circle_size' ,'Post_frequency']

cat_cols = ['Stage_fear', 'Drained_after_socializing']

In [19]:
num_transformer = Pipeline(steps=[
    
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())

])

cat_transformer = Pipeline(steps=[
    
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))

])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

In [20]:
# Encode Target (LabelEncoder)

le = LabelEncoder()

y = le.fit_transform(train['Personality']) #  0 = Extrovert, 1 = Introvert

# Model Building

In [21]:
X = train.drop(columns=['Personality', 'id'])

X_test_final = test.drop(columns=['id'])

In [22]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameter Tuning using GridSearchCV

In [23]:
def run_search_grid(pipe, params, name):
    # Grid Search
    grid = GridSearchCV(pipe, params, cv=5, scoring="accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"\n✅ Finished GridSearch for {name}")
    print("Best Params:", grid.best_params_)
    #print("Best CV Score:", round(grid.best_score_, 3))

    return grid

**Random Forest**

In [24]:
# Random Forest
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])
rf_params = {
    "model__n_estimators": [200, 500, 800],
    "model__max_depth": [None, 10, 20, 30, 40],
    #"model__min_samples_split": [2, 4, 6],
    #"model__min_samples_leaf": [1, 2, 3]
}

In [25]:
# Run Grid Search for Random Forest
grid_rf = run_search_grid(rf_pipe, rf_params, "Random Forest")


✅ Finished GridSearch for Random Forest
Best Params: {'model__max_depth': 10, 'model__n_estimators': 200}


**LightGBM**

In [26]:
# LightGBM
lgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(random_state=42, verbose=-1))
])
lgb_params = {
    "model__n_estimators": [200, 400, 500],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [5, 10, 20, 30],
    #"model__num_leaves": [31, 52, 63, 127]
}

In [27]:
grid_lgb = run_search_grid(lgb_pipe, lgb_params, 'LightGBM')


✅ Finished GridSearch for LightGBM
Best Params: {'model__learning_rate': 0.01, 'model__max_depth': 20, 'model__n_estimators': 400}


**XGBoost**

In [28]:
# XGBoost
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(random_state=42, eval_metric="mlogloss"))
])
xgb_params = {
    "model__n_estimators": [500, 600, 800],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 4, 5, 7],
    #"model__subsample": [0.8, 1.0]
}

In [29]:
grid_xgb = run_search_grid(xgb_pipe, xgb_params, "XGBoost")


✅ Finished GridSearch for XGBoost
Best Params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 800}


**Gradient Boosting**

In [30]:
# Gradient Boosting
gb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])
gb_params = {
    "model__n_estimators": [200, 300, 500],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5],
    #"model__subsample": [0.8, 1.0]
}


In [31]:
grid_gb = run_search_grid(gb_pipe, gb_params, "Gradient Boosting")


✅ Finished GridSearch for Gradient Boosting
Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}


**Decision Tree**

In [32]:
# Decision Tree
dt_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])
dt_params = {
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5, 10],
    #"model__min_samples_leaf": [1, 2, 4]
}

In [33]:
grid_dt = run_search_grid(dt_pipe, dt_params, "Decision Tree")


✅ Finished GridSearch for Decision Tree
Best Params: {'model__max_depth': 5, 'model__min_samples_split': 10}


**Logistic Regression**

In [34]:
# Logistic Regression
lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=500, random_state=42))
])
lr_params = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["lbfgs", "liblinear"]
}

In [35]:
grid_lr = run_search_grid(lr_pipe, lr_params, "Logistic Regression")


✅ Finished GridSearch for Logistic Regression
Best Params: {'model__C': 10, 'model__solver': 'lbfgs'}


**Evaluation Function**

In [36]:
# Evaluation Fuction

def evaluate_model(search, model_name, X, y, X_valid, y_valid):
    best_model = search.best_estimator_
    val_score = cross_val_score(best_model, X, y, cv=5).mean()
    test_score = accuracy_score(y_valid, best_model.predict(X_valid))

    gap = val_score - test_score
    if gap > 0.05 and val_score >= 0.85:
        fit_msg = "🚨 Overfitting"
    elif val_score < 0.70 and test_score < 0.70:
        fit_msg = "⚠️ Underfitting"
    elif abs(gap) <= 0.05 and test_score >= 0.75:
        fit_msg = "✅ Good Fit"
    else:
        fit_msg = "ℹ️ Borderline"

    print(f"\n--- {model_name} ---")
    print("Best Params:", search.best_params_)
    print("Validation Accuracy:", round(val_score, 3))
    print("Test Accuracy:", round(test_score, 3))
    print("Fit Assessment:", fit_msg)

    return {
        "Model": model_name,
        "Validation_Accuracy": round(val_score, 3),
        "Test_Accuracy": round(test_score, 3),
        "Fit_Assessment": fit_msg
    }

# Store model name and fitted GridSearchCV object

In [37]:
grids = [
    ("Random Forest", grid_rf),
    ("Decision Tree", grid_dt),
    ("Logistic Regression", grid_lr),
    ("Gradient Boosting", grid_gb),
    ("XGBoost", grid_xgb),
    ("LightGBM", grid_lgb)
]

results = []
# Loop through each model and collect results
for name, grid in grids:
    res = evaluate_model(grid, name, X_train, y_train, X_valid, y_valid)
    results.append(res)

# Convert results to DataFrame
df_results = pd.DataFrame(results)


--- Random Forest ---
Best Params: {'model__max_depth': 10, 'model__n_estimators': 200}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit

--- Decision Tree ---
Best Params: {'model__max_depth': 5, 'model__min_samples_split': 10}
Validation Accuracy: 0.968
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit

--- Logistic Regression ---
Best Params: {'model__C': 10, 'model__solver': 'lbfgs'}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit

--- Gradient Boosting ---
Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit

--- XGBoost ---
Best Params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 800}
Validation Accuracy: 0.969
Test Accuracy: 0.969
Fit Assessment: ✅ Good Fit

--- LightGBM ---
Best Params: {'model__learning_rate': 0.01, 'model__max_depth': 20, 'model__n_estimators': 400}
Validation A

# Final Prediction

- Among all the models, Lightgbm performs well so I used this model

In [38]:
best_lgb_model = grid_lgb.best_estimator_

test_predictions = best_lgb_model.predict(X_test_final)

# Testing 

In [39]:
test_predictions.shape


(6175,)

In [40]:
X_test_final.shape

(6175, 7)

In [41]:
print(test_predictions[:10])


[0 1 0 0 1 0 0 1 0 1]


In [42]:
single_test = X_test_final.iloc[[0]]  # first row as DataFrame

single_pred = best_lgb_model.predict(single_test)
print(single_pred)

[0]


# Submission

In [43]:
# Example: create a submission DataFrame

submission = pd.DataFrame({
    "id": test["id"],
    "Personality": test_predictions
})

In [44]:
submission.head()

,id,Personality
0,18524,0
1,18525,1
2,18526,0
3,18527,0
4,18528,1


In [45]:
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6175 entries, 0 to 6174
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           6175 non-null   int64
 1   Personality  6175 non-null   int32
dtypes: int32(1), int64(1)
memory usage: 72.5 KB


In [46]:
submission['Personality'] = submission['Personality'].map({0: 'Extrovert', 1: 'Introvert'})

In [47]:
submission.head()

,id,Personality
0,18524,Extrovert
1,18525,Introvert
2,18526,Extrovert
3,18527,Extrovert
4,18528,Introvert


In [48]:
# Save submission
submission.to_csv('submission.csv', index=False)
print("✅ Predictions saved to submission.csv")

✅ Predictions saved to submission.csv


In [49]:
import pickle

# Assume 'best_lgb_model' is your trained LightGBM pipeline
with open("personality_model.pkl", "wb") as f:
    pickle.dump(best_lgb_model, f)
print("✅ Model saved as personality_model.pkl")


✅ Model saved as personality_model.pkl
